# Colored MNIST 3·8 편향 회복 실험 — Colab Pro+

**120개 조건을 한 번에 실행하고, 기본 10분마다 Google Drive에 복원 가능한 백업을 생성하는 노트북**

- **Exposure:** 4, 8, 16, 24, 32 epoch / **Recovery:** 5 epoch
- **p:** 0.5, 0.8, 0.9, 0.99 / **seed:** 42, 43, 44
- **데이터:** MNIST와 EMNIST Digits의 숫자 3·8 / **Batch:** 64
- **측정:** 첫 회복 epoch는 매 step, 나머지는 5 step마다 + 각 epoch 마지막 step
- **저장:** 원본, 학습 진행, 체크포인트, 예측, Q/M/G/F/S/R_Q, CSV, 그래프

**실행 순서:** 메뉴에서 `런타임 → 런타임 유형 변경 → GPU`를 선택 → 아래 셀을 **순서대로** 실행 → 마지막 실행 셀이 실행 중이면 창을 닫아도 돼.

> ⚠️ **보장 범위:** 브라우저를 닫아도 **Colab 서버가 살아 있고 마지막 실험 셀이 실행 중**이면 백업이 계속돼. Colab 런타임이 종료되면 백업도 멈춰. Pro+는 컴퓨팅 유닛이 충분할 때 **최대 24시간** 연속 실행을 지원할 뿐, 완주를 보장하지 않아.
>
> 백업은 로컬 SSD → Drive **주기적 커밋형 스냅샷**이야. 갑작스러운 종료 시 마지막 성공 백업 이후 진도는 손실될 수 있고, 다시 셀을 실행하면 마지막 정상 백업으로 돌아가.
>
> 이 노트북은 원격 제어나 자동 재접속 등 Colab 사용 제한을 우회하는 기능을 사용하지 않아.

공식 안내: [Google Colab FAQ](https://research.google.com/colaboratory/faq.html?hl=ko).

## ① 실행 설정

정식 120개 조건을 실행하려면 `RUN_MODE="full"`을 유지해. 처음 동작 확인을 하고 싶다면 **한 번만** `RUN_MODE="smoke"`로 변경해 실행하고, 확인 후 다시 `full`로 바꿔 실행하면 돼. 두 모드는 Drive 폴더를 분리해 저장해.

In [ ]:
# ===== 사용자 설정 =====
RUN_MODE = "full"       # "smoke" 또는 "full"
BACKUP_INTERVAL_MINUTES = 10  # 기본 10분마다 Drive 스냅샷 백업
DRIVE_FOLDER = "SSHS_ColoredMNIST_ProPlus"

# GPU를 바꾸거나 코드를 수정해 기존 결과를 강제 재개하지 마세요.
assert RUN_MODE in ("smoke", "full")
assert BACKUP_INTERVAL_MINUTES >= 1
print("mode:", RUN_MODE, "backup period:", BACKUP_INTERVAL_MINUTES, "minutes")

## ② GPU 및 Google Drive 연결

Google 계정으로 직접 권한을 승인해야 해. 다른 사람에게 마운트된 Drive 경로나 인증 정보를 공개하지 마.

In [ ]:
import sys, pathlib, subprocess
import torch

assert torch.cuda.is_available(), (
    "CUDA GPU가 없어요. 런타임 → 런타임 유형 변경 → GPU를 선택하고 이 셀을 다시 실행하세요."
)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = pathlib.Path('/content/drive/MyDrive') / DRIVE_FOLDER / RUN_MODE
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print('Drive backup folder:', DRIVE_BASE)

# Colab은 보통 이 라이브러리들을 제공함. PyTorch를 재설치하지 않아서 CUDA 호환성을 유지.
try:
    import filelock, matplotlib, numpy, torchvision
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'filelock>=3.13,<4', 'matplotlib>=3.8,<4', 'numpy>=1.26,<3'])
    import filelock, matplotlib, numpy, torchvision
print('torchvision:', torchvision.__version__, 'numpy:', numpy.__version__)

## ③ 실험 코드 자동 준비

기존 GitHub 저장소에서 `model.py`, `experiment_data.py`, `experiment_io.py`의 **특정 커밋**을 내려받고 Git blob SHA로 검사해. 이후 Colab 실행기와 백업 코드를 노트북에 내장된 원본에서 복원해. GitHub 저장소에 수정·push하지 않아.

In [ ]:
import base64, zlib, hashlib, urllib.request
from pathlib import Path

REPO_CODE = Path('/content/SSHS_ColoredMNIST_Code')
REPO_CODE.mkdir(parents=True, exist_ok=True)
PINNED_COMMIT = 'c4c1042bda3726efd11dacf8f234184f206cf9f4'
BLOB_SHAS = {
    'experiment_data.py': 'bbc0541875362b2a61b54b6901078f61238b5d7c',
    'experiment_io.py': '2202cbbadf2058f7b587bde60923e0db3f5d0e80',
    'model.py': '0c1a8b43fadb3b8ddb19149239814bab10461e45',
}
for filename, expected in BLOB_SHAS.items():
    url = f'https://raw.githubusercontent.com/jyc6704/-1/{PINNED_COMMIT}/{filename}'
    with urllib.request.urlopen(url, timeout=90) as response:
        content = response.read()
    git_sha = hashlib.sha1(f'blob {len(content)}\0'.encode()+content).hexdigest()
    if git_sha != expected:
        raise RuntimeError(f'{filename}: downloaded code is not from pinned commit')
    (REPO_CODE / filename).write_bytes(content)

# 아래 2개 파일은 이 노트북 자체에 압축 형태로 포함되어 있음.
PAYLOADS = {
    'binary_step_experiment_colab.py': (
        '...'
    ),
    'colab_backup_runner.py': (
        'eNqtW/9v4zay/91/BZ+Kh0qpoySLu+LBPR+w3U2vi16ziyS94pBnCLRE26plSUdKSXxB/vc3MyQl6oudZN8tikYSySE5M5z5zHDs'
        'ed6HIuNL9kUW37GM13m8EXLGZJ2zZZpzuWeqqpM9K3KWFTHP2M3NxymDTvG2LNK8YlXBPsr0XjDFVyLbh5PJR6HSdT6bnNqGqpBC'
        'sXS3qyu+zAS7+fn9uz9/f8qTBL4rkbBlViwVK7NaMc7iYrdLqwo+73ieroSqQiD1nuXiofnCUuheL7NUbaDf56u//5PxVSUkSytl'
        'qG04TL0UImd1mRU8gX48T/TKRYIkP+dMirjIcxFX8ETLZNVGABsqnAMWUmaigk2xeyHTVUqLb5bgr3iWLXm8RRYUWSJkgFSvChbL'
        'Ip+y3DLmQaZAj61ksWOJEOVpJrjM03zNUphbAl+LUlH/LbTyDMdUMo23Cun9CDPUpYLV5FWa10Lv9mGTZnqxWnwgryrdCdpitQHu'
        'rGAza1nUuGWRZYzD3qATThtOPM+b0HKiaFVXtRRRBOIpC1kBgbyoeJUWuZpM7De5LrlUwr5vuNpk6dK+/qGK3D4XShMueYVdLNUv'
        '8Gq7qE1dpVnzBqrC27d6WcoiBq1ovuybx2ojBU9gA80H2LGeLgGJ0f5Ni32fUp9/F7mYTD78/NvVL2zO/sRO2MX5O/tn8uP7D7/8'
        '9iX6x+X1zafPV9DhYjKZJGLF6irOiwc/mE0Y/JMCGJU3lENsssRD6BqEqSqA6zte+YGhoDYcFN1HZsyIBwE7/SucB6lJIh9B/nPL'
        '0ND0D6j1Ia02xMewKEXue3LpBYzDOcpLPRz/wYRsWa/gI6i+kH7Gd8uEz7BTiNzyadfBlC09L2iHtbOHdYlb8oFI4O7TtG7EY5Ku'
        'Qd2bPfGq2KVxREodLfeg2c7+prDgPZ62GaMmMyXtAlRI5FW42yap9PWLmt/KGoQkHlNVRcWWXvUyql0JjKGByIgo5ztBE4X4xL5j'
        'Kw8pVinPoqdChWtRlWniB8+eGS/37XaJlUDRcPLBcHLVZcgqpE35ZgtBr3EF5mnjd7/CxCu1z2MfmuFA5oUftB2gUYoy47HwYe4p'
        'bUa3rsCyZpmzQFxbnWdpvvV3qVKg4y03RtiOJ67D9WL5h2H1AfFMmY+DwqTelcqH7sD0XOHJ5ypO0/lPPFMgiDRPQCzzdwEw2Pvf'
        '3AtCkcdFInBbeh1xUe4jbQ5F4isZ2yUkqrKP4rEEkypACUDVzbqg+as0AMe1CoBvXyF/rcKHDlqjIbCb4VmbjihOUVc91YFjGG/q'
        'fPs1B7FdoT2MRCoYdIJpjYqOdMDGoypKHQZKmq7s3M5RZ/81b4XYoSZ5qgS71t7mUspC+ivvpqhlLIABPF+Dm9W+CRUF9HjGnoCt'
        'VipjxwJk2jZ+wz4W4Akr690BD7Q+f0UODwmD/8+/RafNE3LK+oBTo5TozlHQobtJI3Gc7I27c5w4kK+zhNa3FMyeAtgikLVb/Iqz'
        'naHuG6sToYiUr4ilagbCias7OEZTOlsLozrgwH/n2RZQGYCTFjDR2JDdNrDgnsuUA0grwfUDBAFcVCCigXMpSgfEqRARgVVkhA0g'
        'F1nANkGdzVJC2P9O+Y7uAleRE9gvpPPbacV/FrN03BWaI01Xwpp9Gi7XgNl878QLehTMHGT4U0XMAfU0VMgUgIVKFMrf90JgtUet'
        'rW3wcKqm8/DgDZZoZh2lDwh46/WmVxXMZDugWknQHbByuu/rJ/SiqNzHHIQSOYvGbahXEtmnApRz5T2RAJ/PnoiCFABnQYOjqiBe'
        'ByFXUVmo9BGNpfZKoIdxxpUyYPMGkbAx26CeUZTmaRVFvhLZCs4rngdr6l/QU/yHo0IaBAYYG316CbodDCHo8rTVxP0yIH3ZwhpH'
        '9PB5ZILXupZmFJqJiJiOEz9Pmj2DO4mw0ezZQXDOxiqLUEAHKsfsbuE7sMJvHT7JDKdJsKk7M7ou3zHmO1Fx6OWDSYZ1q/TfsAf9'
        'vEOzFOWqY70NWYpt6PHufMHmcyLTs24a2ZleFwtLFqdoOvJcPRAmdbDrQabdbRe4Upxqaka2fc10+nNnrobLGKlFOINhM8zp8NcQ'
        'cPTnjHkU3HnwBH3vZu8W+qklqXJeqk1REUWHGCiETEm/7hbNxyYynLPzwUeNnjpNqI72PE0bQzZmvh2V7tkA7WNhr8AJqwyNtnW5'
        'rfdEakbdWnZpIsGYqcQBB8yxthwuftPQMKEFjdHsMOk7DI0OtRpuQZ+OOjmsD3kJICrxnzyc1Zs5nPS0snmzhjsekYMPSO3ZxQfo'
        '3KyQMQ3QphV07NuE7+TZMCWgHA8JVq5qcUEDLeDwezp2gym7ASGsJQacAYMjarVR4XTACnLMKP8IYEKOfQFs+Wb3gPw88wgN5qm1'
        'YZI/4ElyMLpd2zhQR+cZbcVeG7khXB9skfRriIBh3qAT5A3GEdyeg1uxTI+egK8U+ob4PwCTz9FTZ5q72cX54jnEzXhDgnRs5r1j'
        'bYnT0e7M3VqmYWTToTlFJgbj3Slg6s6otUQvcgrit7RAOp35x+VspN3obWf/IGoN5F+lMi8pTkf7PxT5vchTEDMB4RRdL2hnxpYm'
        'TYR5nvZIkLX8AU8F4FoyTbDSrXAISmjhMt7gQ51Vigmu9pjRSnOFEBlOJAJqgJlg6f5WFGugSoi4PUU4giSAxtLv7NHjAIb3CpDb'
        'EvayCncJ8NorZbHGzJ9lvof+s25fQY5nsG4h73UiKozVvddlnmfSRCI5g1MQmcXrjm7jPc/ShKhEqt7tuNwfJ4Y60e3ptmK6EGzn'
        'PkLF7y/L9XyalfMOtCE/75k2r+PDzcceeAZ7hmvXTsvSPGN0INET0QNIpeX/4gAB8AcVml6/pQL6tgarAgsJDfYOy3ztBUdU1Cx2'
        'nIj1OEyAfQIfGwziY80GXHC7sKF/Ml5Md24h/2x0XaMweBD9dzFQhmLR1DvQWG9qfP/WK0UDh60pjY8qpbhPxcPA1uFHq7Fo7mAZ'
        'kwO8pthHk2k5jLw04Ew3UUSr1zg7KL9e4obWPbXED3t/8RiLsmKfbygcxuQHfJkd2C86hpV3px0o+/399dWCfTFcoLB9xUGaGDAb'
        'CaAKP8Mr0MRYhPIXPZzeo7pgT9acPmuLNn/qWMspXhREZPjmTxadwFevt+aVZxvbbgbEnFFi+OTk3Sy8WD2zX9MfD6zNINSOG2qh'
        'KNmeqPFsfUT6DbJmBQBliFmm+C3Xwilq1Ynx7WSqNcDIxR6yteCnr3uu13MtkOl/ADiOHifMzhFeQd4p31LAnExUgcHxCYuk+Xru'
        '1dXq9H+8YKhdtHILDP0jkKC8ax00BC7walzvokfV6Kv/D57VOoUzZb+IPT2NWJESot5j2YmDa7Jms0FFJ5qpgPNAakIqoXVlNpZY'
        'oHMNk9w9YrCG0z7iK3FjMXuBSxrrXBW5cLZul94mXxuCrxDlaJbljSYXaDRTIxJ348fDGbcXyRbLP3p6ppMab1AyWBlQ0e7XgHxa'
        'URfnv327LdEWuhFhtEfQeNcA/sVbmVnn6b9qHR5W/nBHKG2kTaCsO9MB02xTLzTqTkdgi+CQ2/FLVAOO2cK6Mgk3Lwx1WkrnpCgJ'
        'ph9Rh41Cu3jnKJKg0W0iCsfqPQeHPdhIdtb7lJOVbSNC2tn4xvQEIU8g7nXmHu+MDmQYdxv2NbbnmN/GUd2cJX3RuSKbC6GToYnq'
        'iHfx1v3bW2qiPrJznRUs7TXRqLE03n3KXKvpTnTUhjZa3Hg+c5k+yMGgc8H895sB8mAkWpj+xxYhYSvP9/6gB17NYIawn2bWIMO7'
        'u768uf18fblgf6dahzad24ZHSPIH9Pw8pTt89N5pfroTuwLOo6zz8ABYcAADBfEOEHiPC0BYtyu2dE0PXn6vKxgAA2A6E/aDFhUf'
        'dBkGwGcOxkFn80KH1rVTyNAsGmJG9oCQAr/iUD0OcyM8Q2O6BwRQ1ogOktCBXqLhoE2SDgWG5BxpYQ1D03tZFJk/JIPyGXxtxHfE'
        'cFjBDge3ku06xV6KoL1MMsaqD9J6itHyExSkYHB41yh0mEmZXRS7Ag4fXcQQhy/Bn/zTeD+YroLFOVCNkqj1apU+Uk4lNCeld4XZ'
        '7W7mnLMnusLQFy7tnSi9mktRTTpowkPnFseV2yCF3kAI6qonRNbUYjSJqEs4QrmrQA5+OWXpOsddCLQQakTvR1EHTogrwTmtXF72'
        'ZDp7RM4MBx/3ZSZ51V6lgK86aLOp75xphSbptrp8LK4ydTmgK1YnzemCOBfvCaWolUh+YOb2z+RhkuIhR0TDeF0VoCdpjNeG4+Gn'
        'yQFrudzhQhfhH4C3ySedOLu7mC3e4s40A497s27ciGNtwvil0a0S4sqFq1qDu8T/rCzGQ388CV0Dpll5aBFHLjZ7PdEkUe9D/uU1'
        'rvz3Qm5VyWMBQMWC5cyxyyypJaqPMRlHUjX6SnVHCzm0ORLJK3bn3NbTEG1RhmSJrRE6q1RGO4H1awB9wDpGTegaORfOIwtrAv3G'
        'CV/rdBtsvckv333bgu1vFyYDoEvrnjqmXqcWjrhixx1j49fhouBgQmS4m5ttWpa66q/BbNbvzEZXz3xKjAQv7OINhvvNxnsAjLC8'
        'sUGclHOG+esczBunCk0ELw9plqHho6txABTHsycaDLW3za/XoR689DzvQ1uXijhbA5y62hQyrcxdE2EhOrO6aJOZiSh8Zw9c6bx6'
        '0pREjPovW4FYyHgz6anNJ2oj/Ri7gJ0cMUjHcsSFvu/CwP8oWVQHk04CnSBboFMVIBp1dnKmhEhOzsDCnJzBf25CqA+KmxySE36b'
        '/q8OwGMMPc0gfS+P2RxN+Y4iZ2/RT5GY1EFcUjTddLaeZqjWo3U7upbBLVUGB0yADE5bXD73DOg3gPlhg7q81hmkLYy5AmTLPQNJ'
        'MfDdneCgxkLesA/bKsQepCLEPNgQAtAyQgSPVxJzLy5rOBwPIl1vKhUhiNQXfL3sOZq3drooLaz+tbdbU/Mcq/vOWOf+qy8Fz9V8'
        'b6pXfOe5Ny99t97MMSRWgRBy2NraXJsYas1nb3Ho+muFEaqaA5LbQEiDNy6iLOINPmSFUviXx3EtebzH57zeRea44CvqNs8irGjC'
        '1eqKKhCIz+Xa3r3jcYe3kBQJrGO0A5SAmd6/sAun3oC0qDX1vmf7n9r+u1pR6ddf5+zCs/WyZWEBKU2CHwIE9kV2b29iVSzTEg87'
        '9cYyBqqtp1VHjmRjrNoKy703ce9CaPCIr9YL/gkU9KqofkIrbHVfV385GmoWYO4TFHzCJDx9tAehqKvOPuC9rKv+Tih4dLvhh0gX'
        'FvV6uiU/uit+ibTXGHCIgqy5W4Gk64TwbtYaxBkucmpQ4YzW8uwMt9GUIfkN+5nLhK1r+P+MIl/A4ZJQub3cpFJ5m/LVhX4mVrbl'
        '9cAxAVHslsoIQB1tfDy8Xm6SxNoaRN1LzSdzyQnL9sgvgslE9UWL5VwRm+2AYZXIZiwt0bxzlIQUe9JkRrCnTofAkyn+8U5PPTfv'
        'Ai0hAEn/vBmHkJWSzj701aLG5ZyeIlNPUZ761dRhekGbpiZywbFzY2o4EbzJNBHMznDmkMeEGMiDNVM0pOmsGqWMd1icc6f2yAER'
        '62ttXFhNJkb6WocDWmuzDWyAl+6Fe29v2AffYegJTavdUAN4frtaIAeYR5GWDwsJhjAG7JoJSLG6tjWGdI4j4xbQ+KE2QheF3BjJ'
        'Klnw5xJ9Y7k03lIjmmh+QRF+oaJlWPiUxQ/JHE0PbjuBdczdbp++XE6PRiswBgTpjrm5/fj5t9spQwRglrSsV5jQnF80BxI9f/Ob'
        'jfASzpXNZWdpri+sTDSOJhvU6wEOaqSF6Jo5Ku61TNGF2NzDspge8EAsDt2Gldk4HaWuYfWhZsAQQ6Dn0sXV2D0YbR+rr272g0dv'
        'tEQKE+pNj7/M2ffMrIk+/Df70zke2XNKst8CzvZ0UVkuJkey52DiNQpvetNXOoDdT7fgOAUa3eZzcOz+Fjsgc5O5dwC3o2jD9mIC'
        'BUwosZX1LT35YOYAz867kp2C2RY7AD8tUU1BV9I2RP8FIUxFpXl3BIlaTSGvLOsS7+dUuoZFSoiWnE21sRccYnZDvzACb5eun2eW'
        'MMZhpKJoOalk20QzB2MVsx4ysN2IkdQKlgQgAYv3DzIKwo1oA/NlQipdbtqURqRrMsX6x1Dhzae/fbq6nbL29fby+ldngy6lO+iF'
        'SzKdgeH6CYk5S9GtbdPUZaM5lRwvJexFypwiDWrQnIn05T00tOnrTlyUgzGIErq2ouowEHJRFXka+1ifNg7ATtj3bXml9rrEzrLI'
        'MrwW0vEOSYlqzR0xdJUYsT+Wduz4o38eTtvFnA4WE/RwOgjpgae6pA1tI6zNJ3JT9ucwGBRZDvYGYNDONnt9JUqf2xrBNOWrB9J4'
        'ByUxyJS/FZ9MDhe5NcDF/IBwBLccdiCeOVez7o5DnRkEhL+OyBZiB2MTnw+WoVzSH4hMjhai9Lk0yO8cqli5vL7+fL1gT9W+FD6Q'
        'D8KIcuxR9EKRyn/sCEj04XQCSCVdA0Q2krCIVdSLc7c08H32wPeKNs8g7uUUlqIhRy+AvKilk/6wy8CCUDpaIgPBuLc/A819k8Ye'
        '19RXifNFMR4Q30+OOW8rjo5ID9hh4cUb6l9G6lGdMhtglQP/ps0ElDjRvccrV16qtBqvsvrQ/DAJf+tFMOjolv9/5sExCXELQ1ak'
        'vYhkrLnuSpAqAz20t6UANCDN934xJogAA+EEzYuMp9aAANqFDwOqvcF9s3TU/JDgO0pNS0SP8zxIU56/coM+9NE96fXdgV+FGbc/'
        'ZcaPUyWH49bHr0hG3LgZYDMf9ENtChExbryfU52Q/RWuVATU7O+5w/dyXWMc8oVa/EToaApTU1GUFHEUBc5IrJxAyjQEw0aMKEA3'
        '0FrO9c+B0D+nUiRuXDI+tonVvmq0G8h9HQHU9tMG8n0VjX52yNJZZQVHmCtWHIK7xkqPUumF9UAix7/zRkTXl7++/3T18fK68+ts'
        'Q6snbNSBFH+vpX0WJQKiaIcJuMikA3SsfrMHDLW7hIPmY6bMIYPX5/8HzIPeWg=='
    ),
}
for filename, packed in PAYLOADS.items():
    code = zlib.decompress(base64.b64decode(packed))
    (REPO_CODE / filename).write_bytes(code)
    print(filename, 'prepared (SHA256)', hashlib.sha256(code).hexdigest()[:16])

import ast
for path in REPO_CODE.glob('*.py'):
    ast.parse(path.read_text(encoding='utf-8'), filename=str(path))
print('Fixed research source prepared:', REPO_CODE)


## ④ 전체 실험 실행 — 마지막 셀

**이 셀을 시작한 뒤 학습 출력이 표시되고 셀이 계속 실행 중이면 브라우저를 닫아도 돼.** Colab Pro+ 런타임이 살아 있는 동안 백업 스크립트가 매 10분(설정값)마다 자동 백업해. 실험이 끝나면 마지막으로 한 번 더 백업해.

노트북은 **서버가 종료되었을 때 자동으로 다시 켜거나 새 런타임을 시작하지 않음.** 종료되면 새 런타임에서 동일한 셀을 차례대로 다시 실행해. GPU와 Python 환경이 바뀌면 재개가 거부될 수 있어.

Drive의 `runner_status.json`에서 백업 시각, `view_results/`에서 분석 자료를 확인할 수 있어.

In [ ]:
import subprocess, sys, time
from pathlib import Path

WORKSPACE = Path('/content/SSHS_ColoredMNIST_Workspace')
LOCAL_OUTPUT = WORKSPACE / f'results_{RUN_MODE}'
LOCAL_DATA = WORKSPACE / 'data'
BACKUP_DEST = DRIVE_BASE / 'backup'

research_args = [
    '--device', 'cuda', '--eval-batch-size', '512', '--num-threads', '4',
    '--checkpoint-every-steps', '25', '--late-eval-interval', '5'
]
if RUN_MODE == 'smoke':
    research_args += [
        '--smoke', '--datasets', 'mnist', '--seeds', '42', '--p-values', '0.5', '0.99',
        '--exposure-epochs', '1', '--recovery-epochs', '2'
    ]
else:
    research_args += [
        '--datasets', 'mnist', 'emnist_digits',
        '--seeds', '42', '43', '44',
        '--p-values', '0.5', '0.8', '0.9', '0.99',
        '--exposure-epochs', '4', '8', '16', '24', '32',
        '--recovery-epochs', '5', '--batch-size', '64'
    ]

command = [sys.executable, '-u', str(REPO_CODE/'colab_backup_runner.py'),
    '--repo', str(REPO_CODE),
    '--output', str(LOCAL_OUTPUT),
    '--data-root', str(LOCAL_DATA),
    '--drive-backup', str(BACKUP_DEST),
    '--interval-minutes', str(BACKUP_INTERVAL_MINUTES),
    '--', *research_args]

print('Starting experiment. Browser may close AFTER execution begins.', flush=True)
print('Mode:', RUN_MODE, '| Drive backup:', BACKUP_DEST, flush=True)
print('The next time this notebook is run, the most recent valid backup will be restored.', flush=True)
started = time.monotonic()
result = subprocess.run(command, cwd=str(REPO_CODE))
print('Exit:', result.returncode, '| elapsed (hours):', round((time.monotonic()-started)/3600, 2))
print('View in Drive:', BACKUP_DEST/'view_results')
if result.returncode != 0:
    raise RuntimeError('Experiment stopped. Read runner_status.json, last_runner.log; re-run these cells to resume.')

## ⑤ 완료 후 결과 확인

Drive → `SSHS_ColoredMNIST_ProPlus/full/backup/view_results/` 에서 다음 파일을 열어봐.

- `analysis_brief.md` — 분석 요약 (전체 완료 시)
- `processed/key_results.csv` — 핵심 결과
- `processed/validation_summary.csv` — 조건별 seed 평균·표준편차
- `raw/observations.csv` — step별 측정값 (완료된 조건만)
- `figures/*.png` — 결과 그래프 (완료 시)

**백업 복원용 `blobs/`, `snapshots/`, `latest.json`은 수정하지 마.** 새로운 Colab 런타임에서 ①~④를 다시 실행하면 백업이 자동 복구돼. 아직 미완료면 계속 학습하고, 완료 상태라면 결과를 다시 확인해. Drive 용량 부족·컴퓨팅 유닛 부족·GPU 변경 등의 문제는 자동 해결되지 않아.